# Módulo 1: Clasificación de Severidad en Siniestros Viales

## 1. Contexto y Objetivo
El objetivo de este notebook es construir y comparar modelos de Machine Learning supervisados para predecir si un siniestro vial resultará en **Gravedad** (lesionados o fallecidos) o **Solo Daños**. 

### 2. Control de Fuga de Información (Data Leakage)
Para garantizar que el modelo sea evaluable en tiempo real (al momento del reporte de la emergencia), se excluyen estrictamente:
- Variables posteriores al evento (`fallecido_foraneo`, número de heridos/fallecidos).
- Identificadores y datos geográficos brutos (`codigo`, texto libre de dirección).
- La variable target original (`severidad`).

### 3. Estrategia de Evaluación
Se implementa un **Split Temporal Out-of-Time**:
- **Entrenamiento:** Siniestros ocurridos entre 2016 y 2021.
- **Evaluación (Test):** Siniestros ocurridos en el año 2022 completo.

In [7]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression, Perceptron
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.base import clone
from sklearn.metrics import (average_precision_score, f1_score,
                             precision_recall_curve, roc_auc_score)
from sklearn.pipeline import make_pipeline

In [8]:
df = pd.read_csv('C:\\Users\\santa\\Desktop\\Cositas de Python\\UNI\\IA\\proyectoIA\\data\\clean\\siniestralidad_clean.csv')

C:\Users\santa\AppData\Local\Temp\ipykernel_22508\2175947294.py:1: DtypeWarning: Columns (0: codigo, 1: medio_reporte, 2: direccion_original, 3: direccion_norm) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('C:\\Users\\santa\\Desktop\\Cositas de Python\\UNI\\IA\\proyectoIA\\data\\clean\\siniestralidad_clean.csv')


1. Carga, Limpieza y Feature Engineering

In [9]:
# 1. Creación de Target Binario (1 = Grave, 0 = Solo daños)
df['grave'] = (~df['severidad'].isin(['Solo daños'])).astype(int)

# 2. Tratamiento de variables de Vehículos y manejo consciente de NaN
veh_cols = [c for c in df.columns if c.startswith('veh_')]
df['veh_sin_dato'] = df[veh_cols].isna().any(axis=1).astype(int)
for col in veh_cols:
  df[col] = df[col].fillna(False).astype(int)

# 3. Formato de booleanos
df['es_fin_semana'] = df['es_fin_semana'].astype(int)
df['es_festivo'] = df['es_festivo'].astype(int)

# 4. Transformaciones Temporales (Cíclica para Mes + One-Hot para Día)
df['mes_sin'] = np.sin(2 * np.pi * df['mes'] / 12)
df['mes_cos'] = np.cos(2 * np.pi * df['mes'] / 12)
df = pd.get_dummies(
    df, columns=['dia_semana'], prefix='dia', drop_first=True, dtype=int
)

# 5. Selección de Matriz de Características (X) y Target (y)
feature_cols = (
    ['anio', 'mes_sin', 'mes_cos', 'es_fin_semana', 'es_festivo', 'veh_sin_dato']
    + veh_cols
    + [c for c in df.columns if c.startswith('dia_')]
)

X = df[feature_cols]
y = df['grave']

print(f'Total registros: {X.shape[0]} | Número de predictores: {X.shape[1]}')

Total registros: 124889 | Número de predictores: 20


2. Split Temporal y Escalado

In [10]:
mask_train = df['anio'] < 2022
mask_test = df['anio'] == 2022

X_tr_raw, y_tr = X[mask_train], y[mask_train]
X_te_raw, y_te = X[mask_test], y[mask_test]

scaler = StandardScaler()
X_tr = scaler.fit_transform(X_tr_raw)
X_te = scaler.transform(X_te_raw)

print(
    f'Entrenamiento (2016-2021): {X_tr.shape[0]} registros | % Graves:'
    f' {y_tr.mean():.1%}'
)
print(
    f'Evaluación Test (2022): {X_te.shape[0]} registros | % Graves:'
    f' {y_te.mean():.1%}'
)

Entrenamiento (2016-2021): 106037 registros | % Graves: 36.2%
Evaluación Test (2022): 9979 registros | % Graves: 45.2%


3. Entrenamiento y Evaluación Comparativa

In [11]:
modelos = {
    'Perceptrón': Perceptron(max_iter=1000, random_state=42),
    'Regresión Logística': LogisticRegression(max_iter=1000, random_state=42),
    'k-NN (k=15)': KNeighborsClassifier(n_neighbors=15, n_jobs=-1),
    'SVM Lineal': LinearSVC(dual=False, random_state=42),
    'Árbol de Decisión': DecisionTreeClassifier(max_depth=8, random_state=42),
    'Random Forest': RandomForestClassifier(
        n_estimators=100, max_depth=10, random_state=42, n_jobs=-1
    ),
    'MLP (Red Neuronal)': MLPClassifier(
        hidden_layer_sizes=(32, 16), max_iter=200, random_state=42
    ),
}

resultados = []

In [12]:
for nombre, model in modelos.items():
  model.fit(X_tr, y_tr)
  y_pred = model.predict(X_te)
  report = classification_report(y_te, y_pred, output_dict=True)

  try:
    if hasattr(model, 'predict_proba'):
      auc = roc_auc_score(y_te, model.predict_proba(X_te)[:, 1])
    elif hasattr(model, 'decision_function'):
      auc = roc_auc_score(y_te, model.decision_function(X_te))
    else:
      auc = np.nan
  except Exception:
    auc = np.nan

  resultados.append({
      'Modelo': nombre,
      'Accuracy': report['accuracy'],
      'Precision (Grave)': report['1']['precision'],
      'Recall (Grave)': report['1']['recall'],
      'F1-Score (Grave)': report['1']['f1-score'],
      'ROC-AUC': auc,
  })

df_res = pd.DataFrame(resultados).sort_values(
    by='F1-Score (Grave)', ascending=False
)
display(df_res)

,Modelo,Accuracy,Precision (Grave),Recall (Grave),F1-Score (Grave),ROC-AUC
6,MLP (Red Neuronal),0.754685,0.701974,0.795570,0.745847,0.789068
4,Árbol de Decisión,0.734543,0.704605,0.711628,0.708099,0.790894
2,k-NN (k=15),0.728029,0.713845,0.665781,0.688975,0.778838
5,Random Forest,0.710091,0.736857,0.558804,0.635596,0.793627
3,SVM Lineal,0.667702,0.788916,0.362569,0.496813,0.791127
0,Perceptrón,0.645155,0.697646,0.380731,0.492621,0.668491
1,Regresión Logística,0.661990,0.806989,0.332447,0.470902,0.792833


## Métricas de evaluación: ROC-AUC y PR-AUC

Estas dos métricas **no dependen del umbral de decisión**: evalúan qué tan bien el modelo *ordena* los siniestros del menos al más riesgoso, sin importar a partir de qué puntaje se decida llamar "Grave" a uno. Esto es clave aquí, porque la proporción de graves cambió de 36% (entrenamiento) a 45% (test), y eso distorsiona métricas como el F1 con el umbral por defecto.

### ROC-AUC
- **Interpretación:** si se toma al azar un siniestro grave y uno de solo daños, el ROC-AUC es la probabilidad de que el modelo asigne mayor riesgo al grave.
- **Escala:** 0.5 = azar (no distingue nada) · 1.0 = ordenamiento perfecto.
- **Aquí:** ≈ 0.79-0.80, es decir, en unas 8 de cada 10 parejas (un grave y un solo daños) el modelo pone el mayor riesgo en el grave.
- **Ventaja:** no se ve afectada por el cambio en la proporción de graves entre entrenamiento y test.

### PR-AUC (precisión-recall)
- **Interpretación:** resume el balance entre *precisión* (de los que marco como graves, cuántos lo son) y *recall* (de los graves reales, cuántos detecto) a lo largo de todos los umbrales posibles.
- **Escala:** su valor mínimo de referencia **no es 0.5, sino la proporción de graves en el test** (≈ 0.452). Un modelo al azar obtiene ese valor; uno perfecto, 1.0.
- **Aquí:** ≈ 0.73-0.74, frente a 0.452 del azar.
- **Cuándo es más informativa que ROC-AUC:** cuando interesa sobre todo la clase positiva (los graves) y cuando las clases están desbalanceadas.

### Cómo leerlas en este proyecto

| Métrica | Azar | Mejores modelos | Perceptrón |
|---|---|---|---|
| ROC-AUC | 0.500 | ≈ 0.80 | 0.733 |
| PR-AUC | 0.452 | ≈ 0.74 | 0.681 |

**Limitaciones:** ninguna de las dos dice cuál umbral usar en la práctica (por eso se reporta también el F1 con umbral elegido en un año de validación), y diferencias de ~0.01 entre modelos están dentro del ruido de evaluar con un solo año de test.

In [13]:
a = (df['anio'] <= 2020).values   # entrenamiento
b = (df['anio'] == 2021).values   # validación (elegir umbral)
c = (df['anio'] == 2022).values   # test

def puntaje(m, X_):
    return m.predict_proba(X_)[:, 1] if hasattr(m, 'predict_proba') else m.decision_function(X_)

def evaluar(nombre, modelo, cols):
    m = make_pipeline(StandardScaler(), clone(modelo)).fit(df.loc[a, cols], y[a])
    s_val, s_te = puntaje(m, df.loc[b, cols]), puntaje(m, df.loc[c, cols])
    pr, rc, th = precision_recall_curve(y[b], s_val)
    umbral = th[np.argmax((2 * pr * rc / (pr + rc + 1e-12))[:-1])]
    return {'Modelo': nombre,
            'ROC-AUC': roc_auc_score(y[c], s_te),
            'PR-AUC': average_precision_score(y[c], s_te),
            'F1 (umbral defecto)': f1_score(y[c], m.predict(df.loc[c, cols])),
            'F1 (umbral val)': f1_score(y[c], s_te >= umbral)}

# --- Baselines ---
p = y[c].mean()
print(f"'Todo Grave': F1={2*p/(1+p):.3f}, accuracy={p:.3f} | 'Todo Solo daños': accuracy={1-p:.3f}")
print(f"PR-AUC de un clasificador al azar ≈ {p:.3f}")

# Tabla de consulta: tasa de Grave por combinación exacta de vehículos
llave = (df[veh_cols].values * 2 ** np.arange(len(veh_cols))).sum(axis=1)
tasa = y[a].groupby(llave[a]).mean()
s_lk = pd.Series(llave[c]).map(tasa).fillna(y[a].mean()).values
print(f"Consulta por combinación de vehículos: ROC-AUC={roc_auc_score(y[c], s_lk):.3f} "
      f"| PR-AUC={average_precision_score(y[c], s_lk):.3f}\n")

# --- Comparación principal (los 7 modelos) ---
res = pd.DataFrame([evaluar(n, m, feature_cols) for n, m in modelos.items()]).round(3)
display(res.sort_values('ROC-AUC', ascending=False))

# --- Ablaciones: ¿pesan veh_sin_dato y anio? ---
variantes = {
    'completo': feature_cols,
    'sin veh_sin_dato': [f for f in feature_cols if f != 'veh_sin_dato'],
    'sin anio': [f for f in feature_cols if f != 'anio'],
    'sin ambas': [f for f in feature_cols if f not in ('veh_sin_dato', 'anio')],
}
filas = []
for nom in ['Regresión Logística', 'Árbol de Decisión', 'MLP (Red Neuronal)']:
    for v, cols in variantes.items():
        r = evaluar(nom, modelos[nom], cols)
        r['Variante'] = v
        filas.append(r)
display(pd.DataFrame(filas).round(3)[['Modelo', 'Variante', 'ROC-AUC', 'PR-AUC', 'F1 (umbral val)']])

'Todo Grave': F1=0.623, accuracy=0.452 | 'Todo Solo daños': accuracy=0.548
PR-AUC de un clasificador al azar ≈ 0.452
Consulta por combinación de vehículos: ROC-AUC=0.794 | PR-AUC=0.716



,Modelo,ROC-AUC,PR-AUC,F1 (umbral defecto),F1 (umbral val)
6,MLP (Red Neuronal),0.802,0.744,0.736,0.728
5,Random Forest,0.798,0.743,0.562,0.747
4,Árbol de Decisión,0.795,0.726,0.607,0.739
1,Regresión Logística,0.793,0.735,0.443,0.725
3,SVM Lineal,0.792,0.734,0.448,0.724
2,k-NN (k=15),0.786,0.720,0.608,0.724
0,Perceptrón,0.733,0.681,0.542,0.651


,Modelo,Variante,ROC-AUC,PR-AUC,F1 (umbral val)
0,Regresión Logística,completo,0.793,0.735,0.725
1,Regresión Logística,sin veh_sin_dato,0.786,0.724,0.747
2,Regresión Logística,sin anio,0.793,0.736,0.747
3,Regresión Logística,sin ambas,0.787,0.726,0.748
4,Árbol de Decisión,completo,0.795,0.726,0.739
5,Árbol de Decisión,sin veh_sin_dato,0.795,0.726,0.739
6,Árbol de Decisión,sin anio,0.797,0.727,0.748
7,Árbol de Decisión,sin ambas,0.796,0.725,0.747
8,MLP (Red Neuronal),completo,0.802,0.744,0.728
9,MLP (Red Neuronal),sin veh_sin_dato,0.801,0.743,0.735


En test fuera de muestra (2022), los modelos distinguen siniestros graves de solo daños con ROC-AUC de aprox 0.79-0.80 (azar = 0.5) y PR-AUC de aprox 0.73-0.74 (azar aprox. 0.45). 

Una tabla que solo usa la tasa histórica de graves por combinación de vehículos alcanza ROC-AUC 0.794, igual que la regresión logística: la severidad depende sobre todo de qué vehículos participan, y los modelos no lineales (MLP, Random Forest) mejoran apenas alrededor de 0.01 de AUC. 

Con el umbral elegido en un año de validación, el F1 queda entre 0.72 y 0.75, frente a 0.62 de predecir siempre "Grave". 

Los modelos sirven para priorizar siniestros por riesgo, no para clasificarlos con certeza. 

Se retiraron anio y veh_sin_dato porque reflejan el proceso de registro y no el siniestro, y su ausencia no reduce el AUC.

In [14]:
nombres = [c.replace('veh_', '') for c in veh_cols]
def etiqueta(k):
    return ' + '.join(n for i, n in enumerate(nombres) if (int(k) >> i) & 1) or 'sin vehículo / sin dato'

t = (pd.DataFrame({'k': llave[a], 'grave': y[a].values})
       .groupby('k')['grave'].agg(n='size', pct_grave='mean').reset_index())
t['combinacion'] = t['k'].map(etiqueta)
t['pct_grave'] = (t['pct_grave'] * 100).round(1)
display(t[t['n'] >= 300].sort_values('pct_grave', ascending=False)
          [['combinacion', 'n', 'pct_grave']].head(12))

,combinacion,n,pct_grave
1,moto,9774,86.6
13,bus,1897,73.4
14,moto + bus,924,68.5
25,moto + carga_pesada,544,66.4
31,moto + camioneta_campero + carga_pesada,1208,55.8
5,moto + automovil,16415,47.1
0,sin vehículo / sin dato,21668,38.0
24,carga_pesada,583,36.0
30,camioneta_campero + carga_pesada,1299,20.7
4,automovil,27589,14.6
